#### Инференс ACT

In [ ]:
%%bash
lerobot-record \
  --robot.type=so101_follower \
  --dataset.repo_id=MrAnton/eval_so101_workshop \
  --policy.path=MrAnton/cube_into_buck_policy \
  --episodes=10

In [1]:
from lerobot.cameras.realsense.configuration_realsense import RealSenseCameraConfig
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.datasets.utils import hw_to_dataset_features
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.policies.factory import make_pre_post_processors
from lerobot.teleoperators.so_leader import SO101LeaderConfig, SO101Leader
from lerobot.robots.so_follower import SO101FollowerConfig, SO101Follower
from lerobot.scripts.lerobot_record import record_loop
from lerobot.utils.control_utils import init_keyboard_listener
from lerobot.utils.utils import log_say
from lerobot.utils.visualization_utils import init_rerun
from lerobot.cameras.configs import ColorMode, Cv2Rotation

from lerobot.processor import make_default_processors

NUM_EPISODES = 3
FPS = 30
EPISODE_TIME_SEC = 60
TASK_DESCRIPTION = "Put the red cube into the white bucket "
HF_MODEL_ID = "MrAnton/cube_into_buck_policy"
HF_DATASET_ID = "MrAnton/eval_so101_workshop"

# Create the robot configuration
robot_config = SO101FollowerConfig(
    id="my_awesome_follower_arm",
    cameras={
        "front": RealSenseCameraConfig(
            serial_number_or_name = "231122072775", 
            fps = FPS, 
            width = 640, 
            height = 480, 
            rotation=Cv2Rotation.ROTATE_180) # Optional: fourcc="MJPG" for troubleshooting OpenCV async error.
    },
    port="/dev/ttyACM2",
)

# Initialize the robot
robot = SO101Follower(robot_config)

# Initialize the policy
policy = ACTPolicy.from_pretrained(HF_MODEL_ID)

# Configure the dataset features
action_features = hw_to_dataset_features(robot.action_features, "action")
obs_features = hw_to_dataset_features(robot.observation_features, "observation")
dataset_features = {**action_features, **obs_features}

# Create the dataset
dataset = LeRobotDataset.create(
    repo_id=HF_DATASET_ID,
    fps=FPS,
    features=dataset_features,
    robot_type=robot.name,
    use_videos=True,
    image_writer_threads=4,
)

# Initialize the keyboard listener and rerun visualization
_, events = init_keyboard_listener()
init_rerun(session_name="recording")

# Connect the robot
robot.connect()

preprocessor, postprocessor = make_pre_post_processors(
    policy_cfg=policy,
    pretrained_path=HF_MODEL_ID,
    dataset_stats=dataset.meta.stats,
)

teleop_action_processor, robot_action_processor, robot_observation_processor = make_default_processors()

for episode_idx in range(NUM_EPISODES):
    log_say(f"Running inference, recording eval episode {episode_idx + 1} of {NUM_EPISODES}")

    # Run the policy inference loop
    record_loop(
        robot=robot,
        events=events,
        fps=FPS,
        policy=policy,
        preprocessor=preprocessor,
        postprocessor=postprocessor,
        dataset=dataset,
        control_time_s=EPISODE_TIME_SEC,
        single_task=TASK_DESCRIPTION,
        display_data=True,
        teleop_action_processor=teleop_action_processor,
        robot_action_processor=robot_action_processor,
        robot_observation_processor=robot_observation_processor,
    )

    dataset.save_episode()

# Clean up
robot.disconnect()
dataset.push_to_hub()

/home/anton/miniconda3/envs/workshop_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[2026-03-22T04:09:39Z INFO  re_grpc_server] Listening for gRPC connections on 0.0.0.0:9876. Connect by running `rerun --connect rerun+http://127.0.0.1:9876/proxy`
[2026-03-22T04:09:39Z INFO  winit::platform_impl::linux::x11::window] Guessed window scale factor: 1
[2026-03-22T04:09:39Z WARN  wgpu_hal::vulkan::instance] Unable to find extension: VK_EXT_physical_device_drm
[2026-03-22T04:09:39Z WARN  wgpu_hal::gles::egl] No config found!
[2026-03-22T04:09:39Z WARN  wgpu_hal::gles::egl] EGL says it can present to the window but not natively
[2026-03-22T04:09:39Z WARN  wgpu_hal::gles::adapter] Max vertex attribute stride unknown. Assuming it is 2048
[2026-03-22T04:09:39Z WARN  wgpu_hal::gles::adapter] Max vertex attribut

KeyboardInterrupt: 

Traceback (most recent call last):
  File "/home/anton/miniconda3/envs/workshop_env/bin/rerun", line 6, in <module>
    sys.exit(main())
             ^^^^^^
  File "/home/anton/miniconda3/envs/workshop_env/lib/python3.12/site-packages/rerun_sdk/rerun_cli/__main__.py", line 35, in main
    return subprocess.call([target_path, *sys.argv[1:]])
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/anton/miniconda3/envs/workshop_env/lib/python3.12/subprocess.py", line 391, in call
    return p.wait(timeout=timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/anton/miniconda3/envs/workshop_env/lib/python3.12/subprocess.py", line 1264, in wait
    return self._wait(timeout=timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/anton/miniconda3/envs/workshop_env/lib/python3.12/subprocess.py", line 2053, in _wait
    (pid, sts) = self._try_wait(0)
                 ^^^^^^^^^^^^^^^^^
  File "/home/anton/miniconda3/envs/workshop_env/lib/python3.12/subprocess.py", line 

#### Цикл инференса "напрямую"

In [22]:
import time
import numpy as np
import torch

from lerobot.cameras.realsense.configuration_realsense import RealSenseCameraConfig
from lerobot.cameras.configs import Cv2Rotation
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.policies.factory import make_pre_post_processors
from lerobot.utils.control_utils import predict_action
from lerobot.utils.utils import get_safe_torch_device
from lerobot.robots.so_follower import SO101FollowerConfig, SO101Follower

FPS = 30
TASK_DESCRIPTION = "Put the red cube into the white bucket"
HF_MODEL_ID = "MrAnton/cube_into_buck_policy"

JOINT_KEYS = [
    "shoulder_pan.pos",
    "shoulder_lift.pos",
    "elbow_flex.pos",
    "wrist_flex.pos",
    "wrist_roll.pos",
    "gripper.pos",
]

robot_config = SO101FollowerConfig(
    id="my_awesome_follower_arm",
    cameras={
        "front": RealSenseCameraConfig(
            serial_number_or_name="231122072775",
            fps=FPS,
            width=640,
            height=480,
            rotation=Cv2Rotation.ROTATE_180,
        )
    },
    port="/dev/ttyACM2",
)

robot = SO101Follower(robot_config)
policy = ACTPolicy.from_pretrained(HF_MODEL_ID)

preprocessor, postprocessor = make_pre_post_processors(
    policy_cfg=policy,
    pretrained_path=HF_MODEL_ID,
)

device = get_safe_torch_device(policy.config.device)


def build_policy_observation(obs_raw: dict) -> dict:
    img = obs_raw["front"]
    if not isinstance(img, np.ndarray):
        img = np.asarray(img)

    state = np.array([obs_raw[k] for k in JOINT_KEYS], dtype=np.float32)

    return {
        "observation.images.front": img,
        "observation.state": state,
    }


def action_to_robot_dict(action_values) -> dict:
    if isinstance(action_values, torch.Tensor):
        action_values = action_values.detach().cpu().numpy()

    action_values = np.asarray(action_values, dtype=np.float32).squeeze()

    if action_values.shape[0] != len(JOINT_KEYS):
        raise RuntimeError(
            f"Expected {len(JOINT_KEYS)} action values, got shape {action_values.shape}"
        )

    return {k: float(v) for k, v in zip(JOINT_KEYS, action_values)}


robot.connect()

try:
    print("Starting ACT inference loop. Press Ctrl+C to stop.")

    while True:
        t0 = time.perf_counter()

        obs_raw = robot.get_observation()

        policy_obs = build_policy_observation(obs_raw)

        with torch.inference_mode():
            action_values = predict_action(
                observation=policy_obs,
                policy=policy,
                device=device,
                preprocessor=preprocessor,
                postprocessor=postprocessor,
                use_amp=policy.config.use_amp,
                task=TASK_DESCRIPTION,
                robot_type=robot.robot_type,
            )

        robot_action = action_to_robot_dict(action_values)

        # мягкий шаг, чтобы не дёргать железо резко
        current_state = np.array([obs_raw[k] for k in JOINT_KEYS], dtype=np.float32)
        target_state = np.array([robot_action[k] for k in JOINT_KEYS], dtype=np.float32)

        alpha = 0.15
        blended = (1.0 - alpha) * current_state + alpha * target_state
        safe_action = {k: float(v) for k, v in zip(JOINT_KEYS, blended)}

        robot.send_action(safe_action)

        dt = time.perf_counter() - t0
        time.sleep(max(1.0 / 29-dt, 0.0))

except KeyboardInterrupt:
    print("Stopped by user.")

finally:
    robot.disconnect()

Starting ACT inference loop. Press Ctrl+C to stop.
Stopped by user.
